# Exercise 3 (MapReduce in Practice) [3 points]
---

For this exercise, you are tasked with writing your own MapReduce program in Python and to
run it on the cluster on the provided datasets.   
You may look at the exercise sheet for all the information on the datasets and this task.

In [1]:
# Save variables to access the file locations
posts='C:/Users/sergs/Desktop/TU Wien/2026S/ADBS/Git/adbs/Ex2/posts.csv' # cluster
#posts='../posts.csv' # local

users='C:/Users/sergs/Desktop/TU Wien/2026S/ADBS/Git/adbs/Ex2/users.csv'
#users='../users.csv' # local

comments='C:/Users/sergs/Desktop/TU Wien/2026S/ADBS/Git/adbs/Ex2/comments.csv'
#comments='../comments.csv' # local


In [2]:
%%bash
pip3 install mrjob mr3px six

<3>WSL (9 - Relay) ERROR: CreateProcessCommon:818: execvpe(/bin/bash) failed: No such file or directory


CalledProcessError: Command 'b'pip3 install mrjob mr3px six\n'' returned non-zero exit status 1.

---

- ### **a) Write a MapReduce job with “posts.csv” and "comment.csv" as inputs and following output:**

        (user_questioned, user_answered, user_commented)

where each triple represents a complete question–answer–comment relationship: a user who asked a question, a user who answered that question, and a user who commented on the corresponding answer.

In "posts.csv", the user ids are stored in "owneruserid". Column "posttypeid" stores whether a post is a question (posttypeid = 1) or an answer (posttypeid=2). For answer posts, the column "parentid" stores the id of the corresponding question. In "comments.csv", the answer corresponding to the comment is stored in "postid" and the user who commented is stored in "userid" (You may ignore the comments where the "userid" is empty).

The output should begin with the following lines:

```
8,108,6
8,108,2750
8,319,2750
24,145,919
24,145,2669
24,930,1048
24,16188,919
117,61,117
117,61,61
28524,25392,28524
```

In [ ]:
%%file mymrjob1.py

from mrjob.job import MRJob
from mrjob.step import MRStep
from mrjob.util import log_to_stream
from mr3px.csvprotocol import CsvProtocol
import csv


def parse_id(value):
    """Parse a possibly-float id like '8.0' to int 8. Returns None if empty."""
    value = value.strip()
    if not value:
        return None
    try:
        return int(float(value))
    except (ValueError, TypeError):
        return None


class MyMRJob1(MRJob):

    OUTPUT_PROTOCOL = CsvProtocol

    def set_up_logging(cls, quiet=False, verbose=False, stream=None):
        log_to_stream(name='mrjob', debug=verbose, stream=stream)
        log_to_stream(name='__main__', debug=verbose, stream=stream)

    # ---- Round 1: key by post_id to join answers with their comments ----

    def mapper_join_by_post(self, _, line):
        row = next(csv.reader([line]))

        if len(row) == 20:                        # posts.csv
            if row[0] == 'id':
                return
            post_id   = parse_id(row[0])
            owner     = parse_id(row[13])         # owneruserid
            post_type = parse_id(row[15])         # posttypeid

            if post_id is None or owner is None or post_type is None:
                return

            if post_type == 1:                    # question
                yield str(post_id), ('Q', owner)
            elif post_type == 2:                  # answer
                parent_id = parse_id(row[14])     # parentid = question id
                if parent_id is None:
                    return
                yield str(post_id), ('A', parent_id, owner)

        elif len(row) == 6:                       # comments.csv
            if row[0] == 'id':
                return
            post_id = parse_id(row[2])            # postid (the answer)
            user_id = parse_id(row[5])            # userid (the commenter)

            if post_id is None or user_id is None:
                return
            yield str(post_id), ('C', user_id)

    def reducer_join_by_post(self, post_id, values):
        questions = []
        answers   = []
        comments  = []

        for v in values:
            if   v[0] == 'Q':  questions.append(v)
            elif v[0] == 'A':  answers.append(v)
            elif v[0] == 'C':  comments.append(v)

        # pass question rows through, still keyed by their own post_id
        for q in questions:
            yield post_id, ('Q', q[1])

        # for every (answer, comment) pair, re-key by question_id
        for a in answers:
            question_id  = str(a[1])
            answer_owner = a[2]
            for c in comments:
                yield question_id, ('AC', answer_owner, c[1])

    # ---- Round 2: key by question_id to attach the question owner ----

    def mapper_identity(self, key, value):
        yield key, value

    def reducer_attach_question(self, question_id, values):
        question_owner = None
        pairs = []

        for v in values:
            if v[0] == 'Q':
                question_owner = v[1]
            elif v[0] == 'AC':
                pairs.append((v[1], v[2]))

        if question_owner is not None:
            for answer_owner, commenter in sorted(pairs):
                yield None, (question_owner, answer_owner, commenter)

    def steps(self):
        return [
            MRStep(mapper=self.mapper_join_by_post,
                   reducer=self.reducer_join_by_post),
            MRStep(mapper=self.mapper_identity,
                   reducer=self.reducer_attach_question),
        ]


if __name__ == '__main__':
    MyMRJob1.run()


In [ ]:
! python3 mymrjob1.py  {posts} {comments}

- ### **b) Write a MapReduce job with “posts.csv” and "users.csv" as input and following output:**  

For each user compute the total number of question and answer posts they have created and output the 100 users with highest reputation.

Make sure the output has the following schema:

            displayname, i, reputation, question_count, answer_count

In "posts.csv", the user ids are stored in "owneruserid". Column "posttypeid" stores whether a post is a question (posttypeid = 1) or an answer (posttypeid=2). In "users.csv", the user ids are stored in "id" and their reputations are stored in "reputation".

In case that there are multiple users with the same reputation, you may resolve these conflicts randomly, i.e. pick one of them arbitrarily.

Use a combiner function where possible to reduce the communication cost.

Make sure that your program correctly deals with the header, data types, and possible sparse values.

In case of a correct implementation, the output should contain this text:

```
"Glen_b -Reinstate Monica",1,232889,18,4504
"whuber",2,231500,8,2377
"gung - Reinstate Monica",3,117552,12,1471
"Peter Flom - Reinstate Monica",4,87309,78,2814
"amoeba says Reinstate Monica",5,77182,43,344
```

In [ ]:
%%file mymrjob2.py

from mrjob.job import MRJob
from mrjob.step import MRStep
from mrjob.util import log_to_stream
from mr3px.csvprotocol import CsvProtocol
import csv
import heapq
import logging

log = logging.getLogger(__name__)

K = 100


def parse_id(value):
    """Parse a possibly-float id like '8.0' to int 8. Returns None if empty."""
    value = value.strip()
    if not value:
        return None
    try:
        return int(float(value))
    except (ValueError, TypeError):
        return None


class MyMRJob2(MRJob):

    OUTPUT_PROTOCOL = CsvProtocol

    def set_up_logging(cls, quiet=False, verbose=False, stream=None):
        log_to_stream(name='mrjob', debug=verbose, stream=stream)
        log_to_stream(name='__main__', debug=verbose, stream=stream)

    # ---- Round 1: count posts per user, join with user info ----

    def mapper_count_posts(self, _, line):
        row = next(csv.reader([line]))

        if len(row) == 20:                           # posts.csv
            if row[0] == 'id':
                return
            owner     = parse_id(row[13])            # owneruserid
            post_type = parse_id(row[15])            # posttypeid

            if owner is None or post_type is None:
                return

            if post_type == 1:                       # question
                yield str(owner), ('Q',)
            elif post_type == 2:                     # answer
                yield str(owner), ('A',)

        elif len(row) == 10:                         # users.csv
            if row[0] == 'id':
                return
            user_id      = parse_id(row[0])          # id
            display_name = row[3]                    # displayname
            reputation   = parse_id(row[7])          # reputation

            if user_id is None or reputation is None:
                return
            yield str(user_id), ('U', display_name, reputation)

    def combiner_count_posts(self, user_id, values):
        q_count = 0
        a_count = 0
        for v in values:
            if   v[0] == 'Q':   q_count += 1
            elif v[0] == 'A':   a_count += 1
            elif v[0] == 'QA':  q_count += v[1]; a_count += v[2]
            else:               yield user_id, v     # pass 'U' through
        if q_count > 0 or a_count > 0:
            yield user_id, ('QA', q_count, a_count)

    def reducer_join_user(self, user_id, values):
        display_name = None
        reputation   = 0
        q_count      = 0
        a_count      = 0
        for v in values:
            if   v[0] == 'U':   display_name = v[1]; reputation = v[2]
            elif v[0] == 'Q':   q_count += 1
            elif v[0] == 'A':   a_count += 1
            elif v[0] == 'QA':  q_count += v[1]; a_count += v[2]

        if display_name is not None:
            yield None, (reputation, display_name, q_count, a_count)

    # ---- Round 2: pick top-K users by reputation ----

    def mapper_identity(self, key, value):
        yield key, value

    def reducer_top_k(self, _, values):
        top = heapq.nlargest(K, values, key=lambda x: x[0])
        for rank, (rep, name, qc, ac) in enumerate(top, 1):
            yield None, (name, rank, rep, qc, ac)

    def steps(self):
        return [
            MRStep(mapper=self.mapper_count_posts,
                   combiner=self.combiner_count_posts,
                   reducer=self.reducer_join_user),
            MRStep(mapper=self.mapper_identity,
                   reducer=self.reducer_top_k),
        ]


if __name__ == '__main__':
    MyMRJob2.run()


In [ ]:
!python3 mymrjob2.py {posts} {users}

---
## **Your solution for Exercise 4 will consist of:**  
*  This notebook, filled with your solutions. 
